# Batch Processing: OCR + Layout + Graph cho toàn bộ DocVQA Dataset

Notebook này chạy toàn bộ pipeline qua dataset và lưu kết quả:
1. **OCR**: PaddleOCR extraction
2. **Layout Analysis**: Detect regions (Table, Figure, Form, TextBlock)
3. **Graph Building**: Semantic layout graph với spatial & semantic relations
4. **Export JSON**: Lưu nodes và edges

**Output Format:**
```json
{
  "version": "1.0.0",
  "created_at": "2026-01-12T...",
  "nodes": [
    {
      "node_id": 0,
      "region_type": "text",
      "bbox": [[x1,y1], [x2,y2], [x3,y3], [x4,y4]],
      "score": 0.79,
      "text": "combined text from region"
    }
  ],
  "edges": [
    {"source": 0, "target": 1, "relation": "above", "score": 0.85, "category": "spatial"}
  ],
  "adjacency": {
    "0": [1, 2],
    "1": [0, 2]
  },
  "metadata": {
    "source_image": "12345.png",
    "num_regions": 4,
    "num_nodes": 4,
    "num_edges": 10
  }
}
```

## 1. Import Libraries và Setup

In [14]:
import sys
sys.path.insert(0, '..')

# Force reload modules to get latest changes
import importlib

from pathlib import Path
from datetime import datetime

# Import pipeline components
from src.ocr.ocr_processor import PaddleOCRProcessor
from src.ocr.layout_analyzer import DocumentLayoutAnalyzer
from src.graph.graph_builder import GraphBuilder

# Import utilities
from src.utils import pipeline as pipeline_module
from src.utils import batch_processor as batch_module
from src.utils import statistics_collector as stats_module

# Reload modules to get latest code changes
importlib.reload(pipeline_module)
importlib.reload(batch_module)
importlib.reload(stats_module)

from src.utils.pipeline import FullPipelineProcessor
from src.utils.batch_processor import BatchProcessor
from src.utils.statistics_collector import StatisticsCollector

print("✅ All libraries imported (with reload)!")

✅ All libraries imported (with reload)!


## 2. Configuration

In [15]:
# Dataset paths
IMAGES_FOLDER = Path('../dataset/DocVQA_Images')
OUTPUT_FOLDER = Path('../output/full_pipeline')
SUBSETS = ['train', 'validation', 'test']

# Processing configuration
MAX_IMAGES_PER_SUBSET = 600  # None = process all, or set number like 100
USE_PREPROCESSING = True
MAX_IMAGE_SIZE = 2500

# Create output folder
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("📁 Dataset Folder:", IMAGES_FOLDER)
print("📁 Output Folder:", OUTPUT_FOLDER)
print("📊 Max images per subset:", MAX_IMAGES_PER_SUBSET or "ALL")
print("🔧 Preprocessing:", "ENABLED" if USE_PREPROCESSING else "DISABLED")

📁 Dataset Folder: ..\dataset\DocVQA_Images
📁 Output Folder: ..\output\full_pipeline
📊 Max images per subset: 600
🔧 Preprocessing: ENABLED


## 3. Initialize Pipeline Components

In [16]:
# Test với 1 image
test_image = IMAGES_FOLDER / 'train' / '510.png'

if test_image.exists():
    print(f"Testing pipeline with: {test_image.name}")
    
    # Create temp pipeline
    temp_ocr = PaddleOCRProcessor()
    temp_layout = DocumentLayoutAnalyzer()
    temp_graph = GraphBuilder()
    temp_pipeline = FullPipelineProcessor(temp_ocr, temp_layout, temp_graph)
    
    # Process
    result = temp_pipeline.process_image(image_path=test_image)
    
    if result['success']:
        print("\n✅ Pipeline test successful!")
        print(f"   Nodes: {result['num_nodes']}")
        print(f"   Regions: {result['num_regions']}")
        print(f"   Edges: {result['num_edges']}")
    else:
        print(f"\n❌ Pipeline test failed: {result.get('error')}")
else:
    print(f"⚠️ Test image not found: {test_image}")

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Thach\.paddlex\official_models\PP-OCRv5_server_det`.


Testing pipeline with: 510.png
Đang khởi tạo PaddleOCR engine...


Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Thach\.paddlex\official_models\PP-OCRv5_server_rec`.


-> PaddleOCR đã sẵn sàng!

✅ Pipeline test successful!
   Nodes: 5
   Regions: 5
   Edges: 18


## 3.5. Test Pipeline với 1 Image (Optional)

Test pipeline với 1 ảnh trước khi chạy batch toàn bộ dataset.

In [17]:
# Initialize OCR processor
print("Initializing PaddleOCR...")
ocr_processor = PaddleOCRProcessor(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

# Initialize Layout Analyzer
print("Initializing Layout Analyzer...")
layout_analyzer = DocumentLayoutAnalyzer(
    y_overlap_threshold=0.5,
    line_height_tolerance=0.3,
    max_x_gap_ratio=3.0,
    block_vertical_gap=20,
    block_x_overlap_threshold=0.3
)

# Initialize Graph Builder
print("Initializing Graph Builder...")
graph_builder = GraphBuilder(
    iou_threshold=0.1,
    distance_threshold=200.0,
    projection_threshold=0.3,
    max_neighbors=5,
    min_edge_score=0.2
)

# Initialize Full Pipeline Processor
print("Initializing Pipeline Processor...")
pipeline_processor = FullPipelineProcessor(
    ocr_processor=ocr_processor,
    layout_analyzer=layout_analyzer,
    graph_builder=graph_builder
)

# Initialize Batch Processor
print("Initializing Batch Processor...")
batch_processor = BatchProcessor(pipeline_processor)

print("\n✅ All components initialized!")

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Thach\.paddlex\official_models\PP-OCRv5_server_det`.


Initializing PaddleOCR...
Đang khởi tạo PaddleOCR engine...


Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Thach\.paddlex\official_models\PP-OCRv5_server_rec`.


-> PaddleOCR đã sẵn sàng!
Initializing Layout Analyzer...
Initializing Graph Builder...
Initializing Pipeline Processor...
Initializing Batch Processor...

✅ All components initialized!


## 4. Run Batch Processing

⚠️ **Lưu ý**: Xử lý toàn bộ dataset sẽ mất nhiều thời gian. Bắt đầu với `MAX_IMAGES_PER_SUBSET` nhỏ để test trước.

In [18]:
# Chạy batch processing
print(f"\n{'='*70}")
print("STARTING BATCH PROCESSING")
print(f"{'='*70}\n")

start_time = datetime.now()

# Process dataset using BatchProcessor
stats = batch_processor.process_dataset(
    images_folder=IMAGES_FOLDER,
    output_folder=OUTPUT_FOLDER,
    subsets=SUBSETS,
    max_images_per_subset=MAX_IMAGES_PER_SUBSET,
    skip_existing=True
)

end_time = datetime.now()
elapsed = end_time - start_time

# Print final summary
print(f"\n{'='*70}")
print("FINAL SUMMARY")
print(f"{'='*70}")
print(f"Total Processed: {stats['total_processed']:,}")
print(f"  ✅ Success: {stats['total_success']:,}")
print(f"  ❌ Failed: {stats['total_failed']:,}")
print(f"Time Elapsed: {elapsed}")
print(f"{'='*70}\n")


STARTING BATCH PROCESSING


Processing subset: TRAIN
Found 600 images


Processing train: 100%|██████████| 600/600 [17:09<00:00,  1.72s/it]



TRAIN Summary:
  ✅ Success: 600
  ❌ Failed: 0
  📊 Total: 600

Processing subset: VALIDATION
Found 600 images


Processing validation: 100%|██████████| 600/600 [39:16<00:00,  3.93s/it]



VALIDATION Summary:
  ✅ Success: 600
  ❌ Failed: 0
  📊 Total: 600

Processing subset: TEST
Found 600 images


Processing test:   0%|          | 0/600 [00:00<?, ?it/s]


 Corrupt JSON detected: 10280, re-processing...


Processing test: 100%|██████████| 600/600 [1:19:35<00:00,  7.96s/it]


TEST Summary:
  ✅ Success: 600
  ❌ Failed: 0
  📊 Total: 600

FINAL SUMMARY
Total Processed: 1,800
  ✅ Success: 1,800
  ❌ Failed: 0
Time Elapsed: 2:16:00.753199



## 5. Verify Results

In [19]:
# Count output files
print(f"\n{'='*70}")
print("OUTPUT VERIFICATION")
print(f"{'='*70}\n")

for subset in SUBSETS:
    subset_output = OUTPUT_FOLDER / subset
    if subset_output.exists():
        json_files = list(subset_output.glob('*.json'))
        print(f"{subset:15}: {len(json_files):,} JSON files")
    else:
        print(f"{subset:15}: 0 files (not processed)")

total_files = len(list(OUTPUT_FOLDER.glob('**/*.json')))
print(f"\n{'TOTAL':15}: {total_files:,} JSON files")
print(f"{'='*70}\n")


OUTPUT VERIFICATION

train          : 600 JSON files
validation     : 600 JSON files
test           : 600 JSON files

TOTAL          : 1,801 JSON files



## 6. Inspect Sample Output

In [20]:
# Load and inspect a sample output file
import json

sample_files = list(OUTPUT_FOLDER.glob('**/*.json'))
sample_files = [f for f in sample_files if f.name != 'dataset_statistics.json']

if sample_files:
    sample_file = sample_files[0]
    print(f"📄 Sample file: {sample_file.name}")
    print(f"📁 Path: {sample_file}")
    
    with open(sample_file, 'r', encoding='utf-8') as f:
        sample_data = json.load(f)
    
    print(f"\n{'='*70}")
    print("SAMPLE OUTPUT STRUCTURE")
    print(f"{'='*70}")
    print(f"Version: {sample_data['version']}")
    print(f"Created: {sample_data['created_at']}")
    
    print(f"\nNodes: {len(sample_data['nodes'])}")
    if sample_data['nodes']:
        print(f"  Sample Nodes (top 3):")
        for node in sample_data['nodes'][:3]:
            print(f"    - Node {node['node_id']} ({node['region_type']}): {node['text'][:80]}...")
    
    print(f"\nEdges: {len(sample_data['edges'])}")
    if sample_data['edges']:
        print(f"  Sample Edges (top 5):")
        for i, edge in enumerate(sample_data['edges'][:5], 1):
            print(f"    {i}. Node {edge['source']} → {edge['target']}: {edge['relation']} (score: {edge['score']:.3f}, category: {edge.get('category', 'N/A')})")
    
    print(f"\nMetadata:")
    print(f"  - Source Image: {sample_data['metadata']['source_image']}")
    print(f"  - Num Regions: {sample_data['metadata']['num_regions']}")
    print(f"  - Num Nodes: {sample_data['metadata']['num_nodes']}")
    print(f"  - Num Edges: {sample_data['metadata']['num_edges']}")
    
    print(f"{'='*70}\n")
else:
    print("⚠️ No output files found yet.")

📄 Sample file: 1016.json
📁 Path: ..\output\full_pipeline\test\1016.json

SAMPLE OUTPUT STRUCTURE
Version: 1.0.0
Created: 2026-01-12T14:04:10.563104

Nodes: 3
  Sample Nodes (top 3):
    - Node 0 (text): If you are in agreement with the above terms and conditions, please indicate acc...
    - Node 1 (form): Alan D.Mackenzie Senior Vice President, Sales and Marketing Takeda Pharmaceutica...
    - Node 2 (form): cc: Takeda Chemical Industries, Ltd. 1-1 Doshomachi 4-chome Chuo-ku, Osaka, Japa...

Edges: 6
  Sample Edges (top 5):
    1. Node 0 → 1: above (score: 0.810, category: spatial)
    2. Node 0 → 2: above (score: 0.643, category: spatial)
    3. Node 1 → 0: below (score: 0.810, category: spatial)
    4. Node 1 → 2: above (score: 0.657, category: spatial)
    5. Node 2 → 1: below (score: 0.657, category: spatial)

Metadata:
  - Source Image: 1016.png
  - Num Regions: 3
  - Num Nodes: 3
  - Num Edges: 6



## 7. Collect and Export Statistics

In [21]:
# Collect statistics from all processed files
print("Collecting statistics from all processed files...")
overall_stats = StatisticsCollector.collect_from_folder(OUTPUT_FOLDER)

# Print statistics
StatisticsCollector.print_statistics(overall_stats)

# Save statistics to JSON
stats_file = OUTPUT_FOLDER / 'dataset_statistics.json'
StatisticsCollector.save_statistics(overall_stats, stats_file)

⚠️ Error processing 1016.json: 'ocr'
⚠️ Error processing 1017.json: 'ocr'
⚠️ Error processing 1019.json: 'ocr'
⚠️ Error processing 1021.json: 'ocr'
⚠️ Error processing 1027.json: 'ocr'
⚠️ Error processing 1028.json: 'ocr'
⚠️ Error processing 10280.json: 'ocr'
⚠️ Error processing 10294.json: 'ocr'
⚠️ Error processing 10296.json: 'ocr'
⚠️ Error processing 10297.json: 'ocr'
⚠️ Error processing 10298.json: 'ocr'
⚠️ Error processing 10299.json: 'ocr'
⚠️ Error processing 10300.json: 'ocr'
⚠️ Error processing 10301.json: 'ocr'


⚠️ Error processing 10302.json: 'ocr'
⚠️ Error processing 10304.json: 'ocr'
⚠️ Error processing 10307.json: 'ocr'
⚠️ Error processing 10310.json: 'ocr'
⚠️ Error processing 10312.json: 'ocr'
⚠️ Error processing 10313.json: 'ocr'
⚠️ Error processing 10315.json: 'ocr'
⚠️ Error processing 10320.json: 'ocr'
⚠️ Error processing 10323.json: 'ocr'
⚠️ Error processing 10325.json: 'ocr'
⚠️ Error processing 1050.json: 'ocr'
⚠️ Error processing 1054.json: 'ocr'
⚠️ Error processing 10583.json: 'ocr'
⚠️ Error processing 1085.json: 'ocr'
⚠️ Error processing 11345.json: 'ocr'
⚠️ Error processing 11346.json: 'ocr'


⚠️ Error processing 1172.json: 'ocr'
⚠️ Error processing 1173.json: 'ocr'
⚠️ Error processing 1174.json: 'ocr'
⚠️ Error processing 1175.json: 'ocr'
⚠️ Error processing 1176.json: 'ocr'
⚠️ Error processing 1187.json: 'ocr'
⚠️ Error processing 1188.json: 'ocr'
⚠️ Error processing 1189.json: 'ocr'


⚠️ Error processing 1190.json: 'ocr'
⚠️ Error processing 12552.json: 'ocr'
⚠️ Error processing 12557.json: 'ocr'
⚠️ Error processing 12580.json: 'ocr'
⚠️ Error processing 12582.json: 'ocr'
⚠️ Error processing 12585.json: 'ocr'
⚠️ Error processing 12588.json: 'ocr'
⚠️ Error processing 12714.json: 'ocr'
⚠️ Error processing 12715.json: 'ocr'
⚠️ Error processing 12728.json: 'ocr'
⚠️ Error processing 12740.json: 'ocr'
⚠️ Error processing 12742.json: 'ocr'
⚠️ Error processing 12743.json: 'ocr'
⚠️ Error processing 12744.json: 'ocr'
⚠️ Error processing 12745.json: 'ocr'


⚠️ Error processing 12747.json: 'ocr'
⚠️ Error processing 12749.json: 'ocr'
⚠️ Error processing 12751.json: 'ocr'
⚠️ Error processing 12753.json: 'ocr'
⚠️ Error processing 12754.json: 'ocr'
⚠️ Error processing 1284.json: 'ocr'
⚠️ Error processing 1286.json: 'ocr'
⚠️ Error processing 1288.json: 'ocr'
⚠️ Error processing 1291.json: 'ocr'


...

⚠️ Error processing 27859.json: 'ocr'
⚠️ Error processing 27870.json: 'ocr'
⚠️ Error processing 28057.json: 'ocr'
⚠️ Error processing 28058.json: 'ocr'
⚠️ Error processing 28059.json: 'ocr'
⚠️ Error processing 28060.json: 'ocr'
⚠️ Error processing 28061.json: 'ocr'
⚠️ Error processing 28062.json: 'ocr'
⚠️ Error processing 28063.json: 'ocr'
⚠️ Error processing 28064.json: 'ocr'
⚠️ Error processing 28065.json: 'ocr'
⚠️ Error processing 28066.json: 'ocr'
⚠️ Error processing 283.json: 'ocr'
⚠️ Error processing 28300.json: 'ocr'
⚠️ Error processing 28301.json: 'ocr'


⚠️ Error processing 28304.json: 'ocr'
⚠️ Error processing 28305.json: 'ocr'
⚠️ Error processing 28306.json: 'ocr'
⚠️ Error processing 28307.json: 'ocr'
⚠️ Error processing 28308.json: 'ocr'
⚠️ Error processing 284.json: 'ocr'
⚠️ Error processing 285.json: 'ocr'
⚠️ Error processing 292.json: 'ocr'
⚠️ Error processing 296.json: 'ocr'
⚠️ Error processing 297.json: 'ocr'
⚠️ Error processing 29707.json: 'ocr'
⚠️ Error processing 29709.json: 'ocr'
⚠️ Error processing 29757.json: 'ocr'
⚠️ Error processing 29760.json: 'ocr'
⚠️ Error processing 29763.json: 'ocr'
⚠️ Error processing 29764.json: 'ocr'
⚠️ Error processing 298.json: 'ocr'
⚠️ Error processing 300.json: 'ocr'
⚠️ Error processing 30048.json: 'ocr'
⚠️ Error processing 30049.json: 'ocr'
⚠️ Error processing 30050.json: 'ocr'


⚠️ Error processing 30052.json: 'ocr'
⚠️ Error processing 30185.json: 'ocr'
⚠️ Error processing 30236.json: 'ocr'
⚠️ Error processing 30238.json: 'ocr'
⚠️ Error processing 303.json: 'ocr'
⚠️ Error processing 30356.json: 'ocr'
⚠️ Error processing 30358.json: 'ocr'
⚠️ Error processing 30361.json: 'ocr'
⚠️ Error processing 3039.json: 'ocr'
⚠️ Error processing 304.json: 'ocr'
⚠️ Error processing 3040.json: 'ocr'
⚠️ Error processing 3041.json: 'ocr'
⚠️ Error processing 3042.json: 'ocr'
⚠️ Error processing 3043.json: 'ocr'


⚠️ Error processing 30434.json: 'ocr'
⚠️ Error processing 30437.json: 'ocr'
⚠️ Error processing 3044.json: 'ocr'
⚠️ Error processing 30443.json: 'ocr'
⚠️ Error processing 30444.json: 'ocr'
⚠️ Error processing 3045.json: 'ocr'
⚠️ Error processing 3047.json: 'ocr'
⚠️ Error processing 305.json: 'ocr'
⚠️ Error processing 3062.json: 'ocr'
⚠️ Error processing 3064.json: 'ocr'
⚠️ Error processing 3066.json: 'ocr'
⚠️ Error processing 3069.json: 'ocr'
⚠️ Error processing 307.json: 'ocr'
⚠️ Error processing 3071.json: 'ocr'
⚠️ Error processing 3072.json: 'ocr'
⚠️ Error processing 3074.json: 'ocr'
⚠️ Error processing 3075.json: 'ocr'
⚠️ Error processing 3076.json: 'ocr'
⚠️ Error processing 3077.json: 'ocr'
⚠️ Error processing 3079.json: 'ocr'
⚠️ Error processing 30794.json: 'ocr'
⚠️ Error processing 30798.json: 'ocr'


⚠️ Error processing 30802.json: 'ocr'
⚠️ Error processing 30804.json: 'ocr'
⚠️ Error processing 3081.json: 'ocr'
⚠️ Error processing 31017.json: 'ocr'
⚠️ Error processing 31018.json: 'ocr'
⚠️ Error processing 31020.json: 'ocr'
⚠️ Error processing 31022.json: 'ocr'
⚠️ Error processing 31023.json: 'ocr'
⚠️ Error processing 31025.json: 'ocr'
⚠️ Error processing 31078.json: 'ocr'
⚠️ Error processing 31079.json: 'ocr'
⚠️ Error processing 31080.json: 'ocr'

DATASET STATISTICS
Total Images: 1,800
Total OCR Tokens: 0
Total Layout Regions: 0
Total Graph Edges: 0

Averages per Image:
  Tokens: 0.0
  Regions: 0.0
  Edges: 0.0

✅ Statistics saved to: ..\output\full_pipeline\dataset_statistics.json
